# Task 2: Advanced Support Vector Machine (SVM) Classification
## Telecom Churn Prediction (Competition-Ready Pipeline)

This notebook implements a professional-grade SVM model, specifically addressing the **Class Imbalance** typically found in real-world churn datasets, optimizing Precision & Recall to correctly identify defecting customers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

### 1. Load Dataset & Analyze Target Imbalance
One of the biggest pitfalls in predictive analytics is ignoring class distributions. If 85% of customers don't churn, a naive model can achieve 85% Accuracy simply by guessing "No Churn" every time, resulting in 0% Recall (missing every churning customer).

In [ ]:
df = pd.read_csv('churn-bigml-80.csv')
print(f"Dataset Shape: {df.shape}\n")

print("--- Class Distribution Check ---")
print(df['Churn'].value_counts(normalize=True) * 100)
print("This is highly imbalanced!")

### 2. Preprocessing & Data Scaling
Unlike our previous iteration where we compressed the entire dataset into 2 PCA components (which destroyed predictive variance and caused 0 Precision/Recall), here we will scale all 19 dimensions and allow the SVM to build its plane inside a 19-dimensional space natively.

In [ ]:
# Drop string codes
if 'State' in df.columns:
    df = df.drop(['State'], axis=1)

# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include=['object', 'bool']).columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('Churn', axis=1)
y = df['Churn']

# Features scaling (Mandatory for SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Notice we are keeping the data in its full High-Dimensional State (No PCA compression here)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

### 3. Model Training (Handling the Imbalance)
To achieve competition-level results, we utilize `class_weight='balanced'`. This instructs the SVM algorithm mathematically to heavily penalize mistakes made on the minority "Churn" class, forcing it to hunt them down.

In [ ]:
# Model 1: Baseline Naive RBF (Will struggle due to imbalance)
svm_baseline = SVC(kernel='rbf', probability=True, random_state=42)
svm_baseline.fit(X_train, y_train)

# Model 2: Professionally Tuned RBF (Class Balanced)
svm_balanced = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm_balanced.fit(X_train, y_train)

### 4. Advanced Evaluation Metrics
Watch how the Class-Balanced model violently increases our previously "empty" Recall, meaning we are finally detecting the Churners successfully!

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"AUC:       {roc_auc_score(y_true, y_prob):.4f}\n")

# Baseline evaluate
y_pred_base = svm_baseline.predict(X_test)
y_prob_base = svm_baseline.predict_proba(X_test)[:, 1]
evaluate_model(y_test, y_pred_base, y_prob_base, "Baseline SVM (Unbalanced)")

# Balanced evaluate
y_pred_bal = svm_balanced.predict(X_test)
y_prob_bal = svm_balanced.predict_proba(X_test)[:, 1]
evaluate_model(y_test, y_pred_bal, y_prob_bal, "Professional SVM (Class-Balanced)")

### 5. Statistical Visualizations

In [ ]:
plt.figure(figsize=(15, 6))

# Area Under the Curve comparison
plt.subplot(1, 2, 1)
fpr_base, tpr_base, _ = roc_curve(y_test, y_prob_base)
fpr_bal, tpr_bal, _ = roc_curve(y_test, y_prob_bal)
plt.plot(fpr_base, tpr_base, label=f'Baseline (AUC = {roc_auc_score(y_test, y_prob_base):.3f})')
plt.plot(fpr_bal, tpr_bal, label=f'Class-Balanced (AUC = {roc_auc_score(y_test, y_prob_bal):.3f})', linewidth=2.5)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves: Algorithm Optimization Comparison')
plt.legend()

# High-Yield Confusion Matrix
plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, y_pred_bal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Churn", "Churn"])
disp.plot(ax=plt.gca(), cmap='Blues', colorbar=False)
plt.title("Confusion Matrix: Professional Balanced SVM")

plt.tight_layout()
plt.show()

### 6. Educational 2D Boundary Visualization
You cannot perfectly visualize a 19-dimensional hyperspace model in 2D. However, to fulfill the visual boundaries directive, we compress our dataset into 2 principal components to render a small theoretical 2D model purely for mapping aesthetics.

In [ ]:
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

svm_viz = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_viz.fit(X_train_pca, y_train)

def plot_decision_boundary(clf, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    h = 0.05
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k', alpha=0.6)
    plt.title(title)

plt.figure(figsize=(10, 7))
plot_decision_boundary(svm_viz, X_train_pca, y_train, "Theoretical 2D Boundary Contour (Balanced RBF)")
plt.show()